# Domain-Specific LLM Fine-Tuning with LoRA

Fine-tune an instruction-following language model for a focused domain using Hugging Face Transformers and PEFT/LoRA. The included IT-support examples are illustrative; replace them with reviewed, permissioned data for your chosen domain.

**Recommended:** Google Colab GPU. Training is disabled by default. This is an educational starter, not a production-ready model. Do not include private or confidential data.

## 1. Install dependencies

In [ ]:
%pip install -q "transformers>=4.45,<5" "datasets>=3.0,<4" "peft>=0.13,<1" "accelerate>=1.0,<2" torch pandas

## 2. Imports and configuration

In [ ]:
import os, random, torch, pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling, set_seed
from peft import LoraConfig, get_peft_model, TaskType

set_seed(42)
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./domain-llm-lora-output"
MAX_LENGTH = 512
RUN_TRAINING = False  # Change to True after reviewing data, model license, and runtime.
print("CUDA available:", torch.cuda.is_available())

## 3. Prepare a small illustrative dataset
Replace these examples with accurate, reviewed examples from your target domain. A tiny dataset is only for demonstrating the pipeline.

In [ ]:
examples = [
 {"instruction":"A user cannot connect to office Wi-Fi. What basic checks should they try?",
  "response":"Confirm Wi-Fi is enabled, select the correct network, and re-enter the current password. Toggle Wi-Fi off and on or restart the device. If the issue continues, record the error and device details and escalate to IT. Never ask for the user's password."},
 {"instruction":"How should a user report a suspicious email?",
  "response":"Do not click links, open attachments, or reply. Use the organization's approved phishing-reporting process. Do not share passwords or authentication codes."},
 {"instruction":"A laptop is running slowly. Give safe first troubleshooting steps.",
  "response":"Ask when the slowdown began and whether one application is affected. Save work, close unused applications, and restart if appropriate. Use approved updates and security scans. Do not disable security software or delete unfamiliar system files."},
 {"instruction":"A user forgot their work account password. What should support recommend?",
  "response":"Direct the user to the official password-reset page or approved identity-support process. Never ask for the current password, one-time code, or recovery token. Escalate failed verification through the approved process."},
 {"instruction":"What should an IT support ticket include?",
  "response":"Include a concise summary, affected device or application, reproduction steps, exact error text, start time, business impact, and troubleshooting attempted. Do not include passwords, authentication codes, or unnecessary personal data."},
 {"instruction":"A user receives an MFA prompt they did not initiate. What should they do?",
  "response":"Deny the unexpected prompt and report it using the organization's security process. Follow approved account-protection steps. Never share authentication codes."},
 {"instruction":"A printer is not printing. What details help diagnose the issue?",
  "response":"Check power, paper, displayed errors, and selected printer. Record the operating system, printer model, exact error, when it began, and whether others are affected. Follow local IT procedures before changing shared settings."},
 {"instruction":"A business application shows an error after an update. What should be recorded?",
  "response":"Record application version, device and operating system, exact error, start time, affected users, and reproduction steps. Check approved service-status updates and share logs only through approved channels."},
 {"instruction":"A user asks support to bypass an access restriction. What should support do?",
  "response":"Do not bypass access controls. Explain the approved authorization process, collect the business justification, and route the request to the designated manager or system owner. Record the request for auditability."},
 {"instruction":"How should support handle a suspected malware infection?",
  "response":"Follow the organization's incident-response policy and report the issue promptly. Avoid interacting with suspicious files. Isolation decisions should follow approved procedures. Do not delete evidence or install unapproved cleanup tools."}
]
df = pd.DataFrame(examples)
assert df[["instruction","response"]].notna().all().all()
assert (df["instruction"].str.strip() != "").all() and (df["response"].str.strip() != "").all()
records = df.to_dict(orient="records")
random.Random(42).shuffle(records)
train_records, eval_records = records[:8], records[8:]
display(df.head())
print("Train:", len(train_records), "Evaluation:", len(eval_records))

## 4. Load tokenizer and base model
Check the model card, model license, and compute requirements before using a model in a public or commercial project.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
model.config.use_cache = False
print("Model loaded.")

## 5. Format instruction/response pairs using the chat template

In [ ]:
def format_example(row):
    messages = [
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": row["response"]},
    ]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return f"### Instruction:\n{row['instruction']}\n\n### Response:\n{row['response']}{tokenizer.eos_token}"

print(format_example(train_records[0]))

## 6. Tokenize

In [ ]:
train_ds = Dataset.from_list([{"text": format_example(x)} for x in train_records])
eval_ds = Dataset.from_list([{"text": format_example(x)} for x in eval_records])
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False)
tokenized_train = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized_eval = eval_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
print(tokenized_train)

## 7. Configure LoRA adapters
The listed target module names fit common Qwen transformer blocks. If you change the base model, verify its module names.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. Train (optional)
Training is disabled by default. Set `RUN_TRAINING = True` after checking your GPU memory, dataset quality, and model license. These few examples are not enough for reliable domain performance.

In [ ]:
if RUN_TRAINING:
    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=1,
        save_strategy="epoch",
        eval_strategy="epoch",
        fp16=torch.cuda.is_available(),
        report_to="none",
        gradient_checkpointing=torch.cuda.is_available(),
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )
    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Saved adapter to", OUTPUT_DIR)
else:
    print("Training skipped. Set RUN_TRAINING=True to fine-tune.")

## 9. Generate a response

In [ ]:
def answer_question(question, max_new_tokens=160):
    messages = [{"role":"user","content":question}]
    if getattr(tokenizer, "chat_template", None):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    model.eval()
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(answer_question("What should a user do about an unexpected MFA prompt?"))

## 10. Evaluation checklist

In [ ]:
checklist = [
    "Compare base and fine-tuned models on identical held-out prompts",
    "Review factual accuracy with domain experts",
    "Measure instruction following and completeness",
    "Test out-of-domain prompts and appropriate uncertainty",
    "Check privacy, sensitive-data leakage, and memorization",
    "Track dataset/model versions and training settings",
]
for item in checklist:
    print("☐", item)

## Next steps
1. Replace the demo records with a larger, permissioned, reviewed dataset.
2. Create representative train/validation/test splits and remove duplicates.
3. Compare against the untuned base model with a fixed evaluation set.
4. Track metrics, experiment settings, model version, and dataset version.
5. Consider QLoRA/4-bit training only after the basic LoRA workflow works.

**Limitations:** Fine-tuning does not guarantee factuality, privacy, or security. A tiny dataset can overfit and reduce general capabilities. This is an educational prototype, not a production-ready model.